# Model 2 - VGG Parallel
Notebook train và test VGG Parallel.

In [1]:
# 1. Import và cấu hình
import os
from collections import Counter
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.metrics import classification_report

from metrics import calculate_metrics

DATA_DIR = '../../../data/VN_trash_classification_preprocessing'
CHECKPOINT_DIR = '../checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

EPOCHS = 30
PATIENCE = 7
BATCH_SIZE = 32
LR = 1e-4

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
# 2. Xử lý ảnh và load dữ liệu
normalize = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    normalize,
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize,
])

train_ds = ImageFolder(DATA_DIR + '/train', transform=train_transform)
val_ds = ImageFolder(DATA_DIR + '/val', transform=test_transform)
test_ds = ImageFolder(DATA_DIR + '/test', transform=test_transform)

# Nạp ảnh bằng nhiều tiến trình để GPU không phải chờ CPU
loader_options = {'num_workers': 8, 'pin_memory': True, 'persistent_workers': True}
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, **loader_options)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, **loader_options)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, **loader_options)

class_names = train_ds.classes
print('Device:', device, '| Classes:', class_names)
print(len(train_ds), len(val_ds), len(test_ds))

Device: cuda | Classes: ['Alu', 'Carton', 'Foam_box', 'Milk_box', 'Other', 'PET', 'Paper', 'Paper_cup', 'Plastic_cup']
9542 1685 864


In [3]:
# 3. Tạo VGG Parallel
class VGGParallel(nn.Module):
    def __init__(self, number_of_classes):
        super().__init__()

        # 3 nhánh song song, mỗi nhánh ra 64 kênh
        self.branch1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.branch2 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=2, dilation=2),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.branch3 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=5, padding=2),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )

        # Sau khi nối: 64 + 64 + 64 = 192 kênh, rồi 5 stage kiểu VGG
        self.features = nn.Sequential(
            nn.Conv2d(192, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.6),
            nn.Linear(128, number_of_classes),
        )

    def forward(self, images):
        x = torch.cat([self.branch1(images), self.branch2(images), self.branch3(images)], dim=1)
        return self.classifier(self.features(x))


model = VGGParallel(len(class_names)).to(device)
print(model)

VGGParallel(
  (branch1): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (branch2): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(2, 2), dilation=(2, 2))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (branch3): Sequential(
    (0): Conv2d(3, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (features): Sequential(
    (0): Conv2d(192, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4)

In [4]:
# 4. Loss, optimizer và scheduler
counts = Counter(train_ds.targets)
weights = torch.tensor(
    [len(train_ds) / (len(class_names) * counts[i]) for i in range(len(class_names))],
    dtype=torch.float32,
    device=device,
)

loss_fn = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.01)

In [5]:
# 5. Train và validation
def run_epoch(loader, training=False):
    model.train(training)
    total_loss = 0.0
    correct = 0
    seen = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        seen += labels.size(0)

    return total_loss / seen, correct / seen


best_acc = -1.0
best_state = None
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, True)
    val_loss, val_acc = run_epoch(val_loader)
    scheduler.step()

    if val_acc > best_acc:
        best_acc = val_acc
        best_epoch = epoch + 1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(
        f'Epoch {epoch + 1}/{EPOCHS} | '
        f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2%} | '
        f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2%}'
    )

    # Dừng khi validation accuracy không tăng trong PATIENCE epoch liên tiếp.
    if epochs_without_improvement >= PATIENCE:
        print(f'Early stopping tại epoch {epoch + 1}')
        break

model.load_state_dict(best_state)
print('Best epoch:', best_epoch, '| Best validation accuracy:', f'{best_acc:.2%}')

Epoch 1/30 | Train Loss: 1.9536 | Train Acc: 29.47% | Val Loss: 1.8021 | Val Acc: 35.55%


Epoch 2/30 | Train Loss: 1.7927 | Train Acc: 36.26% | Val Loss: 1.7862 | Val Acc: 37.39%


Epoch 3/30 | Train Loss: 1.6877 | Train Acc: 41.90% | Val Loss: 1.6181 | Val Acc: 43.98%


Epoch 4/30 | Train Loss: 1.5930 | Train Acc: 45.33% | Val Loss: 1.5489 | Val Acc: 48.55%


Epoch 5/30 | Train Loss: 1.4829 | Train Acc: 49.62% | Val Loss: 1.4498 | Val Acc: 49.61%


...

Epoch 26/30 | Train Loss: 0.3811 | Train Acc: 88.38% | Val Loss: 0.5655 | Val Acc: 82.43%


Epoch 27/30 | Train Loss: 0.3685 | Train Acc: 88.96% | Val Loss: 0.5564 | Val Acc: 82.43%


Epoch 28/30 | Train Loss: 0.3572 | Train Acc: 89.24% | Val Loss: 0.5527 | Val Acc: 82.55%


Epoch 29/30 | Train Loss: 0.3547 | Train Acc: 89.55% | Val Loss: 0.5578 | Val Acc: 82.43%


Epoch 30/30 | Train Loss: 0.3375 | Train Acc: 89.53% | Val Loss: 0.5584 | Val Acc: 82.55%
Best epoch: 28 | Best validation accuracy: 82.55%


In [6]:
# 6. Test và đánh giá
model.eval()
true_labels = []
predicted_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predictions.cpu().tolist())

metrics = calculate_metrics(true_labels, predicted_labels)

print()
print('Test Result')
print(f'Test Accuracy: {metrics["accuracy"]:.2%}')
print(f'Macro Precision: {metrics["macro_precision"]:.4f}')
print(f'Macro Recall: {metrics["macro_recall"]:.4f}')
print(f'Macro F1: {metrics["macro_f1"]:.4f}')
print(f'Weighted F1: {metrics["weighted_f1"]:.4f}')
print()
print(classification_report(true_labels, predicted_labels, target_names=class_names, digits=4, zero_division=0))


Test Result
Test Accuracy: 76.04%
Macro Precision: 0.7570
Macro Recall: 0.7569
Macro F1: 0.7477
Weighted F1: 0.7589

              precision    recall  f1-score   support

         Alu     0.8667    0.7879    0.8254        99
      Carton     0.8962    0.8796    0.8879       108
    Foam_box     0.8000    0.4598    0.5839        87
    Milk_box     0.8544    0.8713    0.8627       101
       Other     0.5063    0.8000    0.6202        50
         PET     0.6311    0.6373    0.6341       102
       Paper     0.8019    0.8019    0.8019       106
   Paper_cup     0.8291    0.9238    0.8739       105
 Plastic_cup     0.6273    0.6509    0.6389       106

    accuracy                         0.7604       864
   macro avg     0.7570    0.7569    0.7477       864
weighted avg     0.7717    0.7604    0.7589       864



In [7]:
# 7. Lưu model
torch.save(model.state_dict(), CHECKPOINT_DIR + '/vgg_parallel.pt')